# Polyomino decision classifier on TPU v5e-8

Fine-tune the pinned LFM2.5 Base backbone across **8 TPU devices on one host**. Choose a v5e-8 runtime before running cells. Startup checks require 8 visible TPUs; a single-device Colab TPU runtime won't pass. Availability depends on the notebook host.

This is a separate notebook from the single-device example. It uses JAX data parallelism: each device receives 2 rows per microbatch, with 4 microbatches per optimizer update (64 decisions total at 512 tokens). Parameters and Adam state are replicated; loss and gradients are normalized by the global valid-decision count. The learning rate remains 0.0001. This larger-batch recipe hasn't been measured on TPU hardware yet.

Run cells in order. The notebook kernel never imports JAX or owns the TPU; Python 3.12 child processes do. The 2-update smoke checks every checkpoint tensor before gameplay. The optional long run requires a mounted persistent directory.

The source revision must be available from GitHub. For a local branch, upload the accompanying `minifield-training-tpu-v5e-8.bundle` into `/content` before running; the checkout cell uses that bundle automatically. Downloads still require internet access.

In [ ]:
import json
import math
from pathlib import Path
import subprocess
import sys

CHECKOUT = Path("/content/minifield-training-v5e8")
ROOT = Path("/content/polyomino-tpu-v5e8-smoke")
VENV = ROOT / "venv"
PYTHON = VENV / "bin/python"
SOURCE_REVISION = "e8a9d7d6af83413500b604a0196d198b2c003f8f"
SOURCE_BUNDLE = Path("/content/minifield-training-tpu-v5e-8.bundle")
BASE_REVISION = "9d2be5519834990d30996f878b6771cccbd24f2c"
assert Path("/usr/bin/python3.12").is_file()
ROOT.mkdir(parents=True, exist_ok=True)

def run_child(*command: str, cwd: Path | None = None) -> str:
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        assert process.stdout is not None
        lines = []
        for line in process.stdout:
            print(line, end="", flush=True)
            lines.append(line)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    return "".join(lines)

UV = ROOT / "tooling/bin/uv"
if not UV.is_file():
    run_child(sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
              "--target", str(ROOT / "tooling"), "uv==0.11.30")
assert "uv 0.11.30" in run_child(str(UV), "--version")

In [ ]:
if not (CHECKOUT / ".git").is_dir():
    assert not CHECKOUT.exists(), f"Expected a clean path: {CHECKOUT}"
    clone_source = str(SOURCE_BUNDLE) if SOURCE_BUNDLE.is_file() else "https://github.com/Minifield-Labs/minifield-training.git"
    run_child("git", "clone", clone_source, str(CHECKOUT))
assert not run_child("git", "status", "--porcelain", cwd=CHECKOUT).strip()
# A bundle clone already contains the pin. Fetch only when it's missing.
contains_pin = subprocess.run(["git", "cat-file", "-e", f"{SOURCE_REVISION}^{{commit}}"], cwd=CHECKOUT).returncode == 0
if not contains_pin:
    run_child("git", "fetch", "origin", SOURCE_REVISION, cwd=CHECKOUT)
run_child("git", "checkout", "--detach", SOURCE_REVISION, cwd=CHECKOUT)
assert run_child("git", "rev-parse", "HEAD", cwd=CHECKOUT).strip() == SOURCE_REVISION
assert (CHECKOUT / "examples/polyomino/train.py").is_file()

In [ ]:
if not PYTHON.is_file():
    run_child(str(UV), "venv", "--python", "/usr/bin/python3.12", str(VENV))
run_child(str(UV), "pip", "install", "--python", str(PYTHON),
          "jax[tpu]==0.7.2", "tensorflow-cpu==2.20.0")
run_child(str(UV), "pip", "install", "--python", str(PYTHON),
          ".[numerical,storage,text,hub]", cwd=CHECKOUT)
probe = """
import jax
print('JAX:', jax.__version__, flush=True)
print('Devices:', [(d.id, d.platform, d.device_kind) for d in jax.devices()], flush=True)
assert jax.__version__ == '0.7.2'
assert jax.process_count() == 1, 'This notebook requires a single host'
assert jax.device_count() == jax.local_device_count() == 8, 'Select a runtime exposing 8 TPU devices'
assert all(d.platform == 'tpu' for d in jax.devices()), 'TPU runtime required'
"""
run_child(str(PYTHON), "-c", probe)
assert "--devices" in run_child(str(PYTHON), "-m", "examples.polyomino.train", "--help", cwd=CHECKOUT)

In [ ]:
MODEL_DIR = ROOT / "base-model"
DATASET_CACHE = ROOT / "dataset-cache"
run_child(str(UV), "pip", "install", "--python", str(PYTHON), "huggingface_hub")
download = "from huggingface_hub import snapshot_download; import sys; snapshot_download(repo_id='LiquidAI/LFM2.5-230M-Base', revision=sys.argv[1], allow_patterns=['config.json', 'tokenizer.json', 'model.safetensors'], local_dir=sys.argv[2])"
run_child(str(PYTHON), "-c", download, BASE_REVISION, str(MODEL_DIR), cwd=CHECKOUT)
assert all((MODEL_DIR / name).is_file() for name in ("config.json", "tokenizer.json", "model.safetensors"))
print("Base weights ready. The dataset downloads on the first training call.")

The pinned dataset contains 4,737,585 decisions. Its first load downloads about 2.5 GB of Parquet plus a disk-backed Arrow cache. Allow space for the model and full FP32 checkpoints (about 2.75 GB each).

These settings are shared by smoke, training, and checkpoint evaluation. `--rows` is the global physical batch size. Keep the defaults for the first hardware smoke; changing settings requires fresh checkpoint directories. The old single-device memory measurements don't establish v5e-8 memory use or speed.

In [ ]:
DEVICE_COUNT = 8
ROWS_PER_DEVICE = 2
MICROBATCHES = 4
SEQUENCE_LENGTH = 512
LEARNING_RATE = 0.0001
GLOBAL_ROWS = DEVICE_COUNT * ROWS_PER_DEVICE
DATASET_ROWS = 4_737_585
TOTAL_UPDATES = math.ceil(DATASET_ROWS / (MICROBATCHES * GLOBAL_ROWS))
recipe_args = ["--devices", str(DEVICE_COUNT), "--rows", str(GLOBAL_ROWS),
               "--microbatches", str(MICROBATCHES), "--sequence-length", str(SEQUENCE_LENGTH),
               "--learning-rate", str(LEARNING_RATE)]
recipe_signature = json.dumps({"revision": SOURCE_REVISION, "args": recipe_args}, sort_keys=True)
print(f"{GLOBAL_ROWS} global rows, {ROWS_PER_DEVICE} rows/device, {GLOBAL_ROWS * MICROBATCHES} decisions/update")
print(f"One dataset pass: {TOTAL_UPDATES} updates, including the padded final update")

The smoke executes 2 training updates, saves a checkpoint, reloads it on CPU, and compares every parameter, optimizer moment, step, and cursor with the live replicated state. A mismatch stops execution. Gameplay then runs on one TPU with the saved policy.

A completed smoke can be reused only with the same source revision and recipe. If a smoke was interrupted after writing an unverified checkpoint, choose a fresh `ROOT` and rerun the setup cells.

In [ ]:
CHECKPOINTS = ROOT / "checkpoints-roundtrip"
smoke_run_id = "polyomino-v5e8-smoke-2"
smoke_checkpoint = CHECKPOINTS / "step-00000002" / "manifest.json"
smoke_verified = CHECKPOINTS / "roundtrip-verified"
if not smoke_verified.is_file():
    assert not list(CHECKPOINTS.glob("step-*/manifest.json")), "Use a fresh ROOT for an interrupted, unverified smoke"
    smoke_output = run_child(str(PYTHON), "-u", "-m", "examples.polyomino.train",
              "--model-dir", str(MODEL_DIR), "--dataset-cache", str(DATASET_CACHE),
              "--checkpoint-root", str(CHECKPOINTS), "--run-id", smoke_run_id,
              "--platform", "tpu", *recipe_args, "--max-steps", "2",
              "--checkpoint-every", "2", "--report-every", "1",
              "--verify-checkpoint", "--eval-games", "0", cwd=CHECKOUT)
    assert '"checkpoint_roundtrip_verified":1.0' in smoke_output
    assert smoke_checkpoint.is_file()
    smoke_verified.write_text(recipe_signature + "\n")
assert smoke_checkpoint.is_file()
assert smoke_verified.read_text().strip() == recipe_signature
run_child(str(PYTHON), "-u", "-m", "examples.polyomino.evaluate",
          "--model-dir", str(MODEL_DIR), "--checkpoint", str(smoke_checkpoint.parent),
          "--run-id", smoke_run_id, "--replay-dir", str(CHECKPOINTS / "replays"),
          *recipe_args, "--games", "1", "--max-ticks", "1000", cwd=CHECKOUT)

Set `PERSISTENT_ROOT` to an existing mounted, writable directory to start the optional long run. Use its dedicated v5e-8 subdirectory; the single-device recipe has a different checkpoint identity. The recipe file prevents accidental reuse after changing these settings.

Each call runs for up to 3 hours or the remaining dataset updates. Resume starts at the next unread decision update. Gameplay runs after training on one device, so it doesn't interrupt distributed updates. Checkpoints are written every 5,000 updates and at a normal stop; a disconnected runtime can lose progress since the last saved checkpoint.

In [ ]:
PERSISTENT_ROOT: Path | None = None  # Set to an existing mounted absolute path.
if PERSISTENT_ROOT is None:
    print("Set PERSISTENT_ROOT to run training.")
else:
    assert PERSISTENT_ROOT.is_absolute() and PERSISTENT_ROOT.is_dir()
    assert not PERSISTENT_ROOT.is_relative_to(ROOT), "Use mounted persistent storage"
    long_checkpoints = PERSISTENT_ROOT / "polyomino-v5e8-checkpoints"
    long_checkpoints.mkdir(parents=True, exist_ok=True)
    recipe_file = long_checkpoints / "recipe.json"
    if recipe_file.exists():
        assert recipe_file.read_text().strip() == recipe_signature, "Use a fresh directory for a changed recipe"
    else:
        assert not list(long_checkpoints.glob("step-*/manifest.json")), "Existing checkpoints lack a verified recipe"
        recipe_file.write_text(recipe_signature + "\n")
    run_id = "polyomino-base-decisions-v1-v5e8"
    checkpoints = sorted(long_checkpoints.glob("step-*/manifest.json"))
    completed = json.loads(checkpoints[-1].read_text())["cursor"]["next_batch"] if checkpoints else 0
    assert isinstance(completed, int) and 0 <= completed <= TOTAL_UPDATES
    remaining = TOTAL_UPDATES - completed
    if remaining:
        run_child(str(PYTHON), "-u", "-m", "examples.polyomino.train",
                  "--model-dir", str(MODEL_DIR), "--dataset-cache", str(DATASET_CACHE),
                  "--checkpoint-root", str(long_checkpoints), "--run-id", run_id,
                  "--platform", "tpu", *recipe_args, "--resume-latest", "--max-hours", "3",
                  "--max-steps", str(remaining), "--checkpoint-every", "5000",
                  "--report-every", "50", "--eval-games", "0", cwd=CHECKOUT)
    else:
        print("All decisions have already been visited.")
    checkpoints = sorted(long_checkpoints.glob("step-*/manifest.json"))
    assert checkpoints, "Training produced no checkpoint"
    latest = checkpoints[-1].parent
    run_child(str(PYTHON), "-u", "-m", "examples.polyomino.evaluate",
              "--model-dir", str(MODEL_DIR), "--checkpoint", str(latest),
              "--run-id", run_id, "--replay-dir", str(long_checkpoints / "replays"),
              *recipe_args, "--games", "3", "--max-ticks", "2000", cwd=CHECKOUT)